# Phase 4: Encoder-Decoder Transformer

This notebook explains the **encoder-decoder transformer** designed for autoregressive modeling of DESI spectrum + redshift tokens.

**Why encoder-decoder (not decoder-only)?**

For our two training approaches, encoder-decoder provides clean separation:

| Aspect | Approach A (joint) | Approach B (masked) |
|--------|-------------------|---------------------|
| **Encoder sees** | `[SOS, redshift, s1, s2, ...]` | `[SOS, s1, s2, ...]` (NO redshift!) |
| **Decoder input** | `[SOS, redshift, s1, s2, ...]` (teacher forced) | `[SOS, redshift, s1, s2, ...]` (teacher forced) |
| **Decoder target** | `[redshift, s1, s2, ..., EOS]` | `[redshift, s1, s2, ..., EOS]` |
| **Key property** | Encoder processes redshift bidirectionally | Encoder has **zero** redshift info; decoder predicts redshift from cross-attention only |

**Architecture:**
- **Encoder**: 6 layers, bidirectional self-attention (RoPE), RMSNorm, SwiGLU
- **Decoder**: 6 layers, causal self-attention + cross-attention to encoder, RMSNorm, SwiGLU
- **Vocabulary**: 1288 tokens (1024 spectrum + 256 redshift + 8 special)
- **Shared embeddings**: Input/output weight tying

In [ ]:
import sys
sys.path.insert(0, '..')

import torch
import numpy as np
import matplotlib.pyplot as plt
from src.models.transformer import (
    SpectrumTransformer,
    build_approach_a_sequences,
    build_approach_b_sequences,
    encode_spectrum_token,
    encode_redshift_token,
    SOS_TOKEN, EOS_TOKEN,
    TOTAL_VOCAB_SIZE,
)

%matplotlib inline

## 1. Model Architecture Overview

In [ ]:
model = SpectrumTransformer(
    vocab_size=TOTAL_VOCAB_SIZE,
    d_model=768,
    n_encoder_layers=6,
    n_decoder_layers=6,
    n_heads=12,
    max_seq_len=512,
)

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"Model Configuration:")
print(f"  d_model:           {model.d_model}")
print(f"  n_encoder_layers:  {model.n_encoder_layers}")
print(f"  n_decoder_layers:  {model.n_decoder_layers}")
print(f"  n_heads:           {model.n_heads}")
print(f"  head_dim:          {model.d_model // model.n_heads}")
print(f"  max_seq_len:       {model.max_seq_len}")
print(f"  vocab_size:        {model.vocab_size}")
print(f"")
print(f"Parameter Count:")
print(f"  Total:     {total_params:,}")
print(f"  Trainable: {trainable_params:,}")
print(f"  ~{total_params/1e6:.1f}M parameters")

## 2. Sequence Construction: Approach A vs B

### Approach A (Joint Redshift Prediction)

**Encoder input:** `[SOS, redshift, s1, s2, ..., sN, EOS]`

The encoder processes redshift **bidirectionally** with all spectrum tokens.

### Approach B (Masked Redshift)

**Encoder input:** `[SOS, s1, s2, ..., sN, EOS]`

The encoder has **absolutely no redshift information**. The decoder must predict redshift from cross-attention to the encoder's spectrum-only representation.

In [ ]:
# Example tokens
redshift_tok = encode_redshift_token(torch.tensor(128))
spectrum_toks = encode_spectrum_token(torch.tensor([100, 200, 300, 400, 500]))

# Approach A
enc_a, dec_a, target_a = build_approach_a_sequences(redshift_tok, spectrum_toks)

# Approach B
enc_b, dec_b, target_b = build_approach_b_sequences(redshift_tok, spectrum_toks)

print("Approach A (joint prediction):")
print(f"  Encoder:  {enc_a.tolist()}")
print(f"  Decoder:  {dec_a.tolist()}")
print(f"  Target:   {target_a.tolist()}")
print()
print("Approach B (masked redshift):")
print(f"  Encoder:  {enc_b.tolist()}")
print(f"  Decoder:  {dec_b.tolist()}")
print(f"  Target:   {target_b.tolist()}")

# Verify key property
assert redshift_tok.item() in enc_a, "Approach A encoder must contain redshift"
assert redshift_tok.item() not in enc_b, "Approach B encoder must NOT contain redshift"
print("\n✅ Verified: Approach B encoder has no redshift token!")

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 8))

def plot_seq(ax, seq, title, highlight_idx=None, highlight_color='lightgreen'):
    colors = ['lightgray'] * len(seq)
    if highlight_idx is not None:
        for i in highlight_idx:
            if 0 <= i < len(seq):
                colors[i] = highlight_color
    ax.bar(range(len(seq)), seq, color=colors, edgecolor='black')
    ax.set_title(title, fontsize=10, fontweight='bold')
    ax.set_xlabel('Position')
    ax.set_ylabel('Token ID')
    ax.set_xticks(range(len(seq)))
    for i, v in enumerate(seq):
        if v < 10:
            ax.text(i, v + 20, str(v), ha='center', fontsize=7)

# Approach A
plot_seq(axes[0, 0], enc_a.tolist(), "A: Encoder (sees redshift)", highlight_idx=[1], highlight_color='lightblue')
plot_seq(axes[0, 1], dec_a.tolist(), "A: Decoder Input", highlight_idx=[1], highlight_color='lightblue')
plot_seq(axes[0, 2], target_a.tolist(), "A: Target", highlight_idx=[0], highlight_color='orange')

# Approach B
plot_seq(axes[1, 0], enc_b.tolist(), "B: Encoder (NO redshift)", highlight_color='salmon')
plot_seq(axes[1, 1], dec_b.tolist(), "B: Decoder Input", highlight_idx=[1], highlight_color='lightblue')
plot_seq(axes[1, 2], target_b.tolist(), "B: Target", highlight_idx=[0], highlight_color='orange')

plt.suptitle('Encoder-Decoder Sequence Comparison', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../plots/approach_comparison.png', dpi=150)
plt.show()

## 3. Forward Pass Test

Verify the model can process both approaches and compute loss.

In [ ]:
batch_size = 4

# Build batch for Approach A
enc_a_batch = enc_a.unsqueeze(0).repeat(batch_size, 1)
dec_a_batch = dec_a.unsqueeze(0).repeat(batch_size, 1)
target_a_batch = target_a.unsqueeze(0).repeat(batch_size, 1)

logits_a, loss_a = model(enc_a_batch, dec_a_batch, targets=target_a_batch)

# Build batch for Approach B
enc_b_batch = enc_b.unsqueeze(0).repeat(batch_size, 1)
dec_b_batch = dec_b.unsqueeze(0).repeat(batch_size, 1)
target_b_batch = target_b.unsqueeze(0).repeat(batch_size, 1)

logits_b, loss_b = model(enc_b_batch, dec_b_batch, targets=target_b_batch)

print(f"Approach A:")
print(f"  Encoder shape:  {enc_a_batch.shape}")
print(f"  Decoder shape:  {dec_a_batch.shape}")
print(f"  Logits shape:   {logits_a.shape}")
print(f"  Loss:           {loss_a.item():.4f}")
print()
print(f"Approach B:")
print(f"  Encoder shape:  {enc_b_batch.shape}")
print(f"  Decoder shape:  {dec_b_batch.shape}")
print(f"  Logits shape:   {logits_b.shape}")
print(f"  Loss:           {loss_b.item():.4f}")

## 4. Generation Test

Test autoregressive generation from encoder input.

In [ ]:
# Approach B: encoder has NO redshift
enc_prompt = enc_b.unsqueeze(0)

generated = model.generate(enc_prompt, max_new_tokens=10, temperature=1.0)

print(f"Encoder input:  {enc_prompt[0].tolist()}")
print(f"Generated:      {generated[0].tolist()}")
print(f"Length: {generated.shape[1]} tokens")

# First generated token should be the redshift prediction
first_gen = generated[0, 1].item()  # After SOS
print(f"\nFirst predicted token (redshift): {first_gen}")
print(f"Ground truth redshift token:      {redshift_tok.item()}")

## 5. Summary

The encoder-decoder transformer is ready for training:
- **{total_params/1e6:.1f}M parameters** with {model.n_encoder_layers} encoder + {model.n_decoder_layers} decoder layers
- **Encoder** uses bidirectional attention for rich spectrum representation
- **Decoder** uses causal self-attention + cross-attention to encoder
- **Clean separation** between Approach A and B via encoder input control
- **Approach A**: encoder sees redshift bidirectionally with spectrum
- **Approach B**: encoder sees spectrum ONLY; decoder predicts redshift from cross-attention

Next: Phase 5 (tokenized dataset + Approach A training) and Phase 6 (Approach B training).